# 105 — Delta-ML with Uncertainty-Weighted Blending

**Motivation:** nb97's multi-template delta prediction dominates grand_v8 with 99.9% weight, but some test compounds have high template disagreement (templates disagree on the delta) — this suggests uncertainty. When templates disagree, blending toward a direct LGBM prediction should be safer.

**Strategy:**
1. Same multi-template delta framework as nb97 (Tanimoto [0.35, 0.90], K=10)
2. Compute per-query uncertainty = variance across template predictions (not just delta)
3. Adaptive blending:
   - `adaptive_weight = 1 / (1 + template_variance)`  [range 0 to 1]
   - `final_pred = adaptive_weight * delta_pred + (1 - adaptive_weight) * direct_pred`
   - Low variance (confident templates) → trust delta fully
   - High variance (disagreeing templates) → fall back toward direct LGBM
4. Optionally tune the uncertainty threshold with CV

**Key hypothesis:** The variance proxy is meaningful: compounds with noisy templates genuinely have uncertain delta predictions.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, compute_physchem
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)

In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
props = ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]
print("Computing physchem...", flush=True)
phys_tr = tr["smiles"].map(compute_physchem).tolist()
phys_arr = np.array([[p.get(k,0) or 0 for k in props] for p in phys_tr], dtype=np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")

Computing physchem...


Train 4,139  Test 513  Cliffs 0


In [4]:
# --- Build delta-pair dataset (same as nb97) ---
SIM_LO = 0.35; SIM_HI = 0.90; MAX_PAIRS = 400_000
print("Computing pairwise Tanimoto...", flush=True)
dot_tt = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum = fps_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)
i_idx, j_idx = np.where((tanimoto_tr >= SIM_LO) & (tanimoto_tr <= SIM_HI))
mask_upper = i_idx < j_idx
i_idx, j_idx = i_idx[mask_upper], j_idx[mask_upper]
print(f"Pairs in sim window [{SIM_LO},{SIM_HI}]: {len(i_idx):,}")
rng = np.random.default_rng(SEED)
if len(i_idx) > MAX_PAIRS:
    sel = rng.choice(len(i_idx), MAX_PAIRS, replace=False)
    i_idx, j_idx = i_idx[sel], j_idx[sel]
    print(f"Downsampled to {MAX_PAIRS:,}")

Computing pairwise Tanimoto...


Pairs in sim window [0.35,0.9]: 5,177


In [5]:
# --- Feature engineering (same as nb97) ---
def compress_fp(fp, out_dim=64):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_delta_feats(fp_anchor, fp_query, sim_col, anchor_pec50, phys_diff):
    fp_common = np.minimum(fp_anchor, fp_query).astype(np.float32)
    fp_diff   = np.abs(fp_anchor - fp_query).astype(np.float32)
    c64 = compress_fp(fp_common)
    d64 = compress_fp(fp_diff)
    return np.hstack([c64, d64, sim_col, anchor_pec50[:,None], phys_diff])

sims_ij = tanimoto_tr[i_idx, j_idx][:,None]
phys_diff_ij = phys_arr[j_idx] - phys_arr[i_idx]
F_ij = make_delta_feats(fps_tr[i_idx], fps_tr[j_idx], sims_ij, y_tr[i_idx], phys_diff_ij)
F_ji = make_delta_feats(fps_tr[j_idx], fps_tr[i_idx], sims_ij, y_tr[j_idx], -phys_diff_ij)
F_all = np.vstack([F_ij, F_ji])
y_all = np.concatenate([y_tr[j_idx]-y_tr[i_idx], y_tr[i_idx]-y_tr[j_idx]])
print(f"Delta dataset: {F_all.shape}  delta range [{y_all.min():.2f}, {y_all.max():.2f}]")

print("Training global delta LGBM...", flush=True)
DELTA_LGBM = dict(n_estimators=600, num_leaves=63, learning_rate=0.05,
                  min_child_samples=20, subsample=0.8, colsample_bytree=0.7,
                  reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
delta_model = lgb.LGBMRegressor(**DELTA_LGBM)
delta_model.fit(F_all, y_all, callbacks=[lgb.log_evaluation(-1)])
print("Delta model trained.", flush=True)

Delta dataset: (10354, 137)  delta range [-4.68, 4.68]
Training global delta LGBM...


Delta model trained.


In [6]:
# --- Uncertainty-weighted multi-template prediction ---
K_NEIGHBORS = 10

def uncertainty_delta_predict(fps_query, fps_ref, y_ref, phys_query, phys_ref,
                               sim_matrix, delta_model, direct_preds,
                               sim_lo=SIM_LO, sim_hi=SIM_HI, k=K_NEIGHBORS):
    """
    Multi-template delta prediction with uncertainty-weighted blending.
    Returns: (final_preds, delta_preds, uncertainties, adaptive_weights, n_templates)
    """
    N = len(fps_query)
    delta_preds   = np.full(N, np.nan)   # raw delta-ML prediction (before blending)
    uncertainties = np.full(N, np.nan)   # variance across template predictions
    n_templates   = np.zeros(N, dtype=int)

    for qi in range(N):
        sim_row = sim_matrix[qi]
        cand_mask = (sim_row >= sim_lo) & (sim_row <= sim_hi)
        cand_idx = np.where(cand_mask)[0]
        if len(cand_idx) == 0:
            delta_preds[qi] = direct_preds[qi]
            uncertainties[qi] = 0.0  # no templates = fully trust direct
            continue
        top_k = np.argsort(-sim_row[cand_idx])[:k]
        cand_idx = cand_idx[top_k]
        cand_sims = sim_row[cand_idx]
        n_templates[qi] = len(cand_idx)

        fp_q_rep = np.tile(fps_query[qi:qi+1], (len(cand_idx), 1))
        fp_refs  = fps_ref[cand_idx]
        sims_col = cand_sims[:,None]
        anc_pec50 = y_ref[cand_idx]
        phys_d = phys_query[qi:qi+1] - phys_ref[cand_idx]
        F_k = make_delta_feats(fp_refs, fp_q_rep, sims_col, anc_pec50, phys_d)
        delta_k = delta_model.predict(F_k)
        template_preds = y_ref[cand_idx] + delta_k

        # Sim^2-weighted mean and variance
        weights = cand_sims ** 2
        w_norm = weights / weights.sum()
        w_mean = np.sum(w_norm * template_preds)
        w_var  = np.sum(w_norm * (template_preds - w_mean) ** 2)

        delta_preds[qi]   = w_mean
        uncertainties[qi] = float(w_var)

    # Adaptive blending: alpha = 1 / (1 + variance)
    # High variance -> alpha small -> trust direct more
    # Low variance  -> alpha near 1 -> trust delta fully
    uncertainties_safe = np.where(np.isnan(uncertainties), 0.0, uncertainties)
    adaptive_weights = 1.0 / (1.0 + uncertainties_safe)  # [0, 1]
    # Compounds with no templates: adaptive_weight=1 but delta_pred=direct_pred anyway
    final_preds = adaptive_weights * delta_preds + (1.0 - adaptive_weights) * direct_preds
    final_preds = np.where(np.isnan(delta_preds), direct_preds, final_preds)

    return final_preds, delta_preds, uncertainties, adaptive_weights, n_templates

print(f"Uncertainty-weighted delta function ready (K={K_NEIGHBORS})")

Uncertainty-weighted delta function ready (K=10)


In [7]:
# --- Scaffold 5-fold CV ---
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_unc   = np.full(len(y_tr), np.nan)  # uncertainty-blended prediction
oof_raw   = np.full(len(y_tr), np.nan)  # raw delta prediction (no uncertainty blending)
oof_direct = np.full(len(y_tr), np.nan)
oof_var   = np.full(len(y_tr), np.nan)  # per-compound template variance
oof_aw    = np.full(len(y_tr), np.nan)  # adaptive weight

for fold, (tr_idx, va_idx) in enumerate(splits):
    m_dir = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
    dot_vf = (fps_va @ fps_ft.T).astype(np.float32)
    rs_v = fps_va.sum(1)[:,None]; rs_f = fps_ft.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)

    fp, dp, unc, aw, nt = uncertainty_delta_predict(
        fps_va, fps_ft, y_tr[tr_idx], phys_arr[va_idx], phys_arr[tr_idx],
        sim_vf, delta_model, oof_direct[va_idx]
    )
    oof_unc[va_idx]    = fp
    oof_raw[va_idx]    = dp
    oof_var[va_idx]    = unc
    oof_aw[va_idx]     = aw

    r_dir = rae(y_tr[va_idx], oof_direct[va_idx])
    r_raw = rae(y_tr[va_idx], oof_raw[va_idx])
    r_unc = rae(y_tr[va_idx], oof_unc[va_idx])
    print(f"  fold {fold+1}  direct={r_dir:.4f}  raw_delta={r_raw:.4f}  "
          f"unc_blend={r_unc:.4f}  avg_var={float(np.nanmean(unc)):.4f}  "
          f"avg_aw={float(np.nanmean(aw)):.3f}  avg_tmpl={float(nt.mean()):.1f}",
          flush=True)

m_dir = full_metrics(y_tr, oof_direct, cliff_pairs, "direct_lgbm")
m_raw = full_metrics(y_tr, oof_raw,    cliff_pairs, "raw_delta_ml")
m_unc = full_metrics(y_tr, oof_unc,    cliff_pairs, "uncertainty_blend")
print(f"\nOverall avg template variance: {float(np.nanmean(oof_var)):.4f}")
print(f"Overall avg adaptive weight:   {float(np.nanmean(oof_aw)):.4f}")


=== Scaffold 5-fold CV ===


  fold 1  direct=0.4982  raw_delta=0.2912  unc_blend=0.2927  avg_var=0.0046  avg_aw=0.995  avg_tmpl=1.9


  fold 2  direct=0.5759  raw_delta=0.3193  unc_blend=0.3212  avg_var=0.0044  avg_aw=0.996  avg_tmpl=1.8


  fold 3  direct=0.6021  raw_delta=0.3612  unc_blend=0.3629  avg_var=0.0041  avg_aw=0.996  avg_tmpl=1.7


  fold 4  direct=0.5665  raw_delta=0.3294  unc_blend=0.3315  avg_var=0.0038  avg_aw=0.996  avg_tmpl=1.7


  fold 5  direct=0.6033  raw_delta=0.3461  unc_blend=0.3480  avg_var=0.0040  avg_aw=0.996  avg_tmpl=1.7


  [direct_lgbm] RAE=0.5643 MAE=0.5134 R2=0.5991 r=0.7740 rho=0.7268 tau=0.5345
  [raw_delta_ml] RAE=0.3266 MAE=0.2972 R2=0.8178 r=0.9060 rho=0.8754 tau=0.7237
  [uncertainty_blend] RAE=0.3285 MAE=0.2988 R2=0.8170 r=0.9056 rho=0.8751 tau=0.7230

Overall avg template variance: 0.0042
Overall avg adaptive weight:   0.9959


In [8]:
# --- Calibrate uncertainty scale: sweep gamma in adaptive_weight = 1/(1 + gamma*var) ---
print("\nSweeping uncertainty scale gamma...", flush=True)
best_gamma, best_rae_v = 1.0, full_metrics(y_tr, oof_unc)["RAE"]
for gamma in [0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]:
    aw_g = 1.0 / (1.0 + gamma * np.where(np.isnan(oof_var), 0.0, oof_var))
    blended = aw_g * oof_raw + (1.0 - aw_g) * oof_direct
    blended = np.where(np.isnan(oof_raw), oof_direct, blended)
    mask = np.isfinite(blended)
    r = rae(y_tr[mask], blended[mask])
    print(f"  gamma={gamma:.2f}  RAE={r:.4f}  avg_weight={float(aw_g.mean()):.3f}")
    if r < best_rae_v:
        best_rae_v, best_gamma = r, gamma

print(f"\nBest gamma={best_gamma:.2f}  OOF RAE={best_rae_v:.4f}")
aw_best = 1.0 / (1.0 + best_gamma * np.where(np.isnan(oof_var), 0.0, oof_var))
oof = aw_best * oof_raw + (1.0 - aw_best) * oof_direct
oof = np.where(np.isnan(oof_raw), oof_direct, oof)
m_best = full_metrics(y_tr, oof, cliff_pairs, f"best_gamma={best_gamma:.2f}")
print("\n" + pd.DataFrame([m_dir, m_raw, m_unc, m_best],
                           index=["direct","raw_delta","gamma=1",f"gamma={best_gamma:.2f}"]).round(4).to_string())


Sweeping uncertainty scale gamma...


  gamma=0.10  RAE=0.3268  avg_weight=1.000
  gamma=0.25  RAE=0.3271  avg_weight=0.999
  gamma=0.50  RAE=0.3276  avg_weight=0.998
  gamma=1.00  RAE=0.3285  avg_weight=0.996
  gamma=2.00  RAE=0.3302  avg_weight=0.992
  gamma=5.00  RAE=0.3348  avg_weight=0.981
  gamma=10.00  RAE=0.3413  avg_weight=0.966
  gamma=20.00  RAE=0.3517  avg_weight=0.942
  gamma=50.00  RAE=0.3718  avg_weight=0.895

Best gamma=0.10  OOF RAE=0.3268
  [best_gamma=0.10] RAE=0.3268 MAE=0.2974 R2=0.8177 r=0.9060 rho=0.8754 tau=0.7237

               RAE     MAE      R2  Pearson  Spearman  Kendall
direct      0.5643  0.5134  0.5991   0.7740    0.7268   0.5345
raw_delta   0.3266  0.2972  0.8178   0.9060    0.8754   0.7237
gamma=1     0.3285  0.2988  0.8170   0.9056    0.8751   0.7230
gamma=0.10  0.3268  0.2974  0.8177   0.9060    0.8754   0.7237


In [9]:
# --- Final test predictions ---
print("\nFitting final direct LGBM on all train...", flush=True)
m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

dot_tet = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_tet / np.maximum(rs_te + rs_tr_v - dot_tet, 1e-6)
phys_te = np.array([[p.get(k,0) or 0 for k in props]
                     for p in te["smiles"].map(compute_physchem)], dtype=np.float32)

print("Running uncertainty-weighted delta on test...", flush=True)
te_fp, te_dp, te_unc, te_aw, n_tmpl_te = uncertainty_delta_predict(
    fps_te, fps_tr, y_tr, phys_te, phys_arr,
    sim_te_tr, delta_model, te_direct
)
print(f"Test: avg_var={float(np.nanmean(te_unc)):.4f}  avg_aw={float(te_aw.mean()):.3f}  "
      f"avg_tmpl={n_tmpl_te.mean():.1f}")

aw_te = 1.0 / (1.0 + best_gamma * np.where(np.isnan(te_unc), 0.0, te_unc))
te_preds = aw_te * te_dp + (1.0 - aw_te) * te_direct
te_preds = np.where(np.isnan(te_dp), te_direct, te_preds)
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_delta_uncertainty.npy", oof)
np.save(DATA_PROCESSED/"te_oof_delta_uncertainty.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"105_delta_with_uncertainty.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb105 OOF RAE = {m_best['RAE']:.4f} ***")


Fitting final direct LGBM on all train...


Running uncertainty-weighted delta on test...


Test: avg_var=0.0626  avg_aw=0.948  avg_tmpl=3.1
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\105_delta_with_uncertainty.csv
Test: min=2.80 med=4.72 max=5.74

*** nb105 OOF RAE = 0.3268 ***
